In [1]:
import sys
sys.path.append("../src")

In [2]:
from db_connection import DatabaseConnector
from queries import NBAReceiverQuaries
from data_loader import NBADataLoader
from statistical_tests import StatisticalTester


In [3]:
db = DatabaseConnector(env_path="../.env")
engine = db.create_connection()


In [4]:
loader = NBADataLoader(engine)

df_agility = loader.load_agility_data(NBAReceiverQuaries.query_agility)
df_intrinsic = loader.load_intrinsic_data(NBAReceiverQuaries.query_intrinsic)


In [5]:
df_agility.head()


,season_years,period_group,player_id,fullname,agility
0,2020-2021,past_period,1036,Stephen Curry,0.400000
1,2020-2021,past_period,2561,Damian Lillard,0.370000
2,2020-2021,past_period,120,Giannis Antetokounmpo,0.341564
3,2020-2021,past_period,1480,Julius Randle,0.324000
4,2020-2021,past_period,3935,Joel Embiid,0.300000


In [6]:
df_agility.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   season_years  41 non-null     object 
 1   period_group  41 non-null     object 
 2   player_id     41 non-null     int64  
 3   fullname      41 non-null     object 
 4   agility       41 non-null     float64
dtypes: float64(1), int64(1), object(3)
memory usage: 1.7+ KB


In [7]:
df_intrinsic.head()

,season_years,period_group,champion_club,player_id,fullname,age,experience,intrinsic_talent
0,2020-2021,past_period,Atlanta Hawks,727,Clint Capela,26.0,6,0.230769
1,2020-2021,past_period,Atlanta Hawks,898,John Collins,23.0,3,0.130435
2,2020-2021,past_period,Atlanta Hawks,3845,Kris Dunn,26.0,4,0.153846
3,2020-2021,past_period,Atlanta Hawks,4013,Bruno Fernando,22.0,1,0.045455
4,2020-2021,past_period,Atlanta Hawks,3205,Kevin Huerter,22.0,2,0.090909


In [8]:
df_intrinsic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 182 entries, 0 to 181
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   season_years      182 non-null    object 
 1   period_group      182 non-null    object 
 2   champion_club     182 non-null    object 
 3   player_id         182 non-null    int64  
 4   fullname          182 non-null    object 
 5   age               182 non-null    float64
 6   experience        182 non-null    int64  
 7   intrinsic_talent  182 non-null    float64
dtypes: float64(2), int64(2), object(4)
memory usage: 11.5+ KB


In [9]:
tester = StatisticalTester(alpha=0.05)

results = []


In [10]:
agility_result = tester.run_hypothesis_test(
    df=df_agility,
    value_column="agility",
    hypothesis_name="Agility of Top 20 MVP Candidates",
    group_column="period_group"
)

In [11]:
tester.print_report(agility_result)

Hypothesis Test Report: Agility of Top 20 MVP Candidates

Value Column:
agility

Sample Sizes:
Past period: 23
Recent period: 18

Means:
Past mean: 0.35874697154208784
Recent mean: 0.3669407438782334
Mean difference: 0.008193772336145544

Normality Test - Past Period:
{'test': 'Shapiro-Wilk', 'statistic': 0.9610091175422126, 'p_value': 0.4840469065315078, 'is_normal': True}

Normality Test - Recent Period:
{'test': 'Shapiro-Wilk', 'statistic': 0.9072326044624808, 'p_value': 0.07690352283298738, 'is_normal': True}

Selected Test:
welch

Test Result:
{'test': 'Welch independent t-test', 'alternative': 'recent > past', 'statistic': 0.7564315021749917, 'p_value': 0.22705799940436178, 'is_significant': False}

Description:
از آنجا که مقدار p-value برابر با 0.2271 است و از سطح معناداری alpha = 0.05 بزرگ‌تر یا مساوی می‌باشد، فرض صفر رد نمی‌شود.
بنابراین شواهد آماری کافی وجود ندارد که نتیجه بگیریم مقدار متغیر مورد بررسی در دوره اخیر بیشتر از دوره گذشته است.


In [12]:
intrinsic_result = tester.run_hypothesis_test(
    df=df_intrinsic,
    value_column="intrinsic_talent",
    hypothesis_name="Comparison of intrinsic talent between players of past and recent periods",
    group_column="period_group"
)

In [13]:
tester.print_report(intrinsic_result)

Hypothesis Test Report: Comparison of intrinsic talent between players of past and recent periods

Value Column:
intrinsic_talent

Sample Sizes:
Past period: 72
Recent period: 110

Means:
Past mean: 0.16233394210178834
Recent mean: 0.20830947442313213
Mean difference: 0.0459755323213438

Normality Test - Past Period:
{'test': 'Shapiro-Wilk', 'statistic': 0.9005309493178766, 'p_value': 3.213795685583113e-05, 'is_normal': False}

Normality Test - Recent Period:
{'test': 'Shapiro-Wilk', 'statistic': 0.9369473361725715, 'p_value': 5.742119254170805e-05, 'is_normal': False}

Selected Test:
mannwhitney

Test Result:
{'test': 'Mann-Whitney U test', 'alternative': 'recent > past', 'statistic': 4862.0, 'p_value': 0.004732788822589459, 'is_significant': True}

Description:
از آنجا که مقدار p-value برابر با 0.0047 است و از سطح معناداری alpha = 0.05 کمتر می‌باشد، فرض صفر رد می‌شود.
بنابراین شواهد آماری معناداری وجود دارد که نشان می‌دهد مقدار متغیر مورد بررسی در دوره اخیر بیشتر از دوره گذشته است.
